In [1]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from utils.data_loader_hai import load_hai_train
from scipy.stats import chi2
from scipy.stats import binomtest

In [2]:
# -----------------------
# TranAD
# -----------------------
TranAD_scores = pd.read_csv("../Results/HAI TranAD/scores.csv")
TranAD_CV = pd.read_csv("../Results/HAI TranAD/results_cross_validation.csv")

TranAD_threshold = TranAD_CV["mean_threshold"].iloc[0]
TranAD_scores = TranAD_scores.to_numpy().ravel()


# -----------------------
# DeepSVDD
# -----------------------
DeepSVDD_scores = pd.read_csv("../Results/HAI DeepSVDD/scores.csv")
DeepSVDD_CV = pd.read_csv("../Results/HAI DeepSVDD/results_cross_validation.csv")

DeepSVDD_threshold = DeepSVDD_CV["mean_threshold"].iloc[0]
DeepSVDD_scores = DeepSVDD_scores.to_numpy().ravel()



# -----------------------
# COPOD
# -----------------------
COPOD_scores = pd.read_csv("../Results/HAI COPOD/scores.csv")
COPOD_CV = pd.read_csv("../Results/HAI COPOD/results_cross_validation.csv")

COPOD_threshold = COPOD_CV["mean_threshold"].iloc[0]
COPOD_scores = COPOD_scores.to_numpy().ravel()


# -----------------------
# COUTA
# -----------------------
COUTA_scores = pd.read_csv("../Results/HAI COUTA/scores.csv")
COUTA_CV = pd.read_csv("../Results/HAI COUTA/results_cross_validation.csv")

COUTA_threshold = COUTA_CV["mean_threshold"].iloc[0]
COUTA_scores = COUTA_scores.to_numpy().ravel()


# -----------------------
# LOF
# -----------------------
LOF_scores = pd.read_csv("../Results/HAI LOF/scores.csv")
LOF_CV = pd.read_csv("../Results/HAI LOF/results_cross_validation.csv")

LOF_threshold = LOF_CV["mean_threshold"].iloc[0]
LOF_scores = LOF_scores.to_numpy().ravel()


# -----------------------
# TcnED
# -----------------------
TcnED_scores = pd.read_csv("../Results/HAI TcnED/scores.csv")
TcnED_CV = pd.read_csv("../Results/HAI TcnED/results_cross_validation.csv")

TcnED_threshold = TcnED_CV["mean_threshold"].iloc[0]
TcnED_scores = TcnED_scores.to_numpy().ravel()

In [3]:
# Get predictions from scores and threshold
TranAD_predictions = (TranAD_scores>TranAD_threshold).astype(int)
DeepSVDD_predictions = (DeepSVDD_scores > DeepSVDD_threshold).astype(int)
COPOD_predictions = (COPOD_scores > COPOD_threshold).astype(int)
COUTA_predictions = (COUTA_scores > COUTA_threshold).astype(int)
LOF_predictions = (LOF_scores > LOF_threshold).astype(int)
TcnED_predictions = (TcnED_scores > TcnED_threshold).astype(int)

In [4]:
x_train, x_test, y_test = load_hai_train()

In [5]:
TranAD_correct = TranAD_predictions == y_test
DeepSVDD_correct = DeepSVDD_predictions == y_test
COPOD_correct = COPOD_predictions == y_test
COUTA_correct = COUTA_predictions == y_test
LOF_correct = LOF_predictions == y_test
TcnED_correct = TcnED_predictions == y_test

In [6]:
predictions_dict = {
    "TranAD": TranAD_correct,
    "DeepSVDD": DeepSVDD_correct,
    "COPOD": COPOD_correct,
    "COUTA": COUTA_correct,
    "LOF": LOF_correct,
    "TcnED": TcnED_correct,
}

results = []

for (model1_name, model1_predictions), (model2_name, model2_predictions) in combinations(predictions_dict.items(), 2):

    # Building contingency table
    model1_1_model2_0 = 0
    model1_1_model2_1 = 0
    model1_0_model2_0 = 0
    model1_0_model2_1 = 0

    for i in range(len(model1_predictions)):
        if model1_predictions[i] == 1 and model2_predictions[i] == 1:
            model1_1_model2_1 += 1
        elif model1_predictions[i] == 1 and model2_predictions[i] == 0:
            model1_1_model2_0 += 1
        elif model1_predictions[i] == 0 and model2_predictions[i] == 1:
            model1_0_model2_1 += 1
        elif model1_predictions[i] == 0 and model2_predictions[i] == 0:
            model1_0_model2_0 += 1
        else:
            print("Unhandled case")
    a = model1_1_model2_1
    b = model1_1_model2_0
    c = model1_0_model2_1
    d = model1_0_model2_0

    # mcnemar() uses this for doing McNemar formula: n1, n2 = table[0, 1], table[1, 0] (so b and c)
    # !! not same order as McNemar OG paper !!
    table = [[a, b],
             [c, d]]

    print(table)

    result = mcnemar(table, exact=False)

    print("b+c = ", b+c)

    results.append({
        "Model 1": model1_name,
        "Model 2": model2_name,
        "p-value": result.pvalue,

    })
    print(f"{model1_name} vs {model2_name}")
    print(f"p-value: {result.pvalue:.3e}")
    print(f"xSquared: {result.statistic:.3e}")
    print("-" * 40)

results_df = pd.DataFrame(results)
print(results_df)

results_df.to_csv("McNemar_test_HAI.csv", index=False)

[[562, 124798], [3894, 346]]
b+c =  128692
manual p-value: 0.000e+00
TranAD vs DeepSVDD
p-value: 0.000e+00
xSquared: 1.136e+05
xSquared: 1.136e+05
----------------------------------------
[[120422, 4938], [466, 3774]]
b+c =  5404
manual p-value: 0.000e+00
TranAD vs COPOD
p-value: 0.000e+00
xSquared: 3.699e+03
xSquared: 3.699e+03
----------------------------------------
[[125360, 0], [42, 4198]]
b+c =  42
manual p-value: 2.509e-10
TranAD vs COUTA
p-value: 2.509e-10
xSquared: 4.002e+01
xSquared: 4.002e+01
----------------------------------------
[[124595, 765], [1082, 3158]]
b+c =  1847
manual p-value: 1.941e-13
TranAD vs LOF
p-value: 1.941e-13
xSquared: 5.406e+01
xSquared: 5.406e+01
----------------------------------------
[[125347, 13], [92, 4148]]
b+c =  105
manual p-value: 2.698e-14
TranAD vs TcnED
p-value: 2.698e-14
xSquared: 5.794e+01
xSquared: 5.794e+01
----------------------------------------
[[809, 3647], [120079, 5065]]
b+c =  123726
manual p-value: 0.000e+00
DeepSVDD vs COPOD


In [7]:
# Latex table
alpha = 0.05

models = list(predictions_dict.keys())

# init matrix with blanks on diagonal
sig_matrix = {m: {n: "\\Blank" for n in models} for m in models}

# fill from results_df
for _, row in results_df.iterrows():
    m1 = row["Model 1"]
    m2 = row["Model 2"]
    p = row["p-value"]

    sig = "\\Sig" if p < alpha else "\\NS"

    sig_matrix[m1][m2] = sig
    sig_matrix[m2][m1] = sig  # symmetric

# print LaTeX table
print("\\textbf{" + "} & \\textbf{".join(models) + "}\\\\")
print("\\hline")

for m1 in models:
    row = [f"\\textbf{{{m1}}}"]
    for m2 in models:
        row.append(sig_matrix[m1][m2])
    print(" & ".join(row) + " \\\\")

\textbf{TranAD} & \textbf{DeepSVDD} & \textbf{COPOD} & \textbf{COUTA} & \textbf{LOF} & \textbf{TcnED}\\
\hline
\textbf{TranAD} & \Blank & \Sig & \Sig & \Sig & \Sig & \Sig \\
\textbf{DeepSVDD} & \Sig & \Blank & \Sig & \Sig & \Sig & \Sig \\
\textbf{COPOD} & \Sig & \Sig & \Blank & \Sig & \Sig & \Sig \\
\textbf{COUTA} & \Sig & \Sig & \Sig & \Blank & \Sig & \Sig \\
\textbf{LOF} & \Sig & \Sig & \Sig & \Sig & \Blank & \Sig \\
\textbf{TcnED} & \Sig & \Sig & \Sig & \Sig & \Sig & \Blank \\
